# Part 23: Inference Serving — PagedAttention, Continuous Batching, and Speculation

Notebook 21 derived that **decode is memory-bandwidth-bound and becomes compute-bound only around
batch ~300**. Everything in this notebook follows from that one sentence.

If throughput requires large batches, then a serving system's entire job is **keeping the batch
full**. And the two things that stop it are:

1. **Memory fragmentation** — the KV cache is allocated per request, and wasteful allocation means
   fewer concurrent requests fit.
2. **Serialization** — requests of different lengths finishing at different times, with new ones
   unable to join.

We will build the solutions to both from scratch, plus speculative decoding, and assemble them
into a working toy server.

**What you'll build**
1. Why naive batching wastes most of the GPU
2. **Continuous batching**: an iteration-level scheduler
3. **PagedAttention**: a block table, with copy-on-write
4. **Radix prefix caching**: sharing KV across requests
5. **Chunked prefill**: resolving the TTFT/TPOT conflict
6. **Speculative decoding**, with a proof that it preserves the output distribution
7. Preemption, scheduling policies, and disaggregation
8. An end-to-end toy server with benchmarks

In [2]:
import math
import random
import sys
import time
from collections import defaultdict
from dataclasses import dataclass, field

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, '..')

torch.manual_seed(0)
random.seed(0)

## 1. The problem with naive batching

The obvious way to batch: collect `B` requests, pad them to the same length, run them together
until all finish, then take the next `B`.

Two things go wrong, and both are severe.

**Padding waste.** Requests have wildly different lengths. Real traffic has a long tail — a few
requests generate thousands of tokens while most generate a few dozen. Padding to the maximum
means computing on padding.

**Head-of-line blocking.** The batch cannot retire until its *slowest* member finishes. Nine
requests that need 20 tokens wait for one that needs 2,000. Meanwhile new arrivals queue behind a
batch that is 90% idle slots.

In [3]:
def sample_output_lengths(n, mean=120, seed=0):
    """
    Realistic output-length distribution: lognormal, long-tailed.

    This shape is what makes static batching so bad -- the mean is modest but
    the maximum is far out.
    """
    rng = random.Random(seed)
    return [max(1, int(rng.lognormvariate(math.log(mean), 1.1))) for _ in range(n)]


lengths = sample_output_lengths(512)
print(f"output lengths over 512 requests:")
print(f"  mean {sum(lengths)/len(lengths):.0f}, median {sorted(lengths)[256]}, "
      f"max {max(lengths)}")

BATCH = 32
static_steps = 0
useful = 0
for i in range(0, len(lengths), BATCH):
    group = lengths[i:i + BATCH]
    static_steps += max(group)           # the batch runs until its slowest member
    useful += sum(group)

capacity = static_steps * BATCH
print(f"\nStatic batching at batch {BATCH}:")
print(f"  steps executed:       {static_steps:,}")
print(f"  token-slots consumed: {capacity:,}")
print(f"  useful tokens:        {useful:,}")
print(f"  UTILIZATION:          {useful/capacity:.1%}")
print(f"\nOver {1-useful/capacity:.0%} of the compute is spent on finished or padded")
print("sequences. And notebook 21 showed that batch size is what buys")
print("throughput -- so this is throughput thrown away.")

output lengths over 512 requests:
  mean 212, median 134, max 1791

Static batching at batch 32:
  steps executed:       16,103
  token-slots consumed: 515,296
  useful tokens:        108,618
  UTILIZATION:          21.1%

Over 79% of the compute is spent on finished or padded
sequences. And notebook 21 showed that batch size is what buys
throughput -- so this is throughput thrown away.


## 2. Continuous batching

The fix (Orca, 2022) is to schedule at **iteration** granularity rather than request granularity.
After *every* forward pass:

- Sequences that emitted their stop token **leave** the batch and free their memory.
- Waiting requests **join** immediately, filling the vacated slots.

The batch composition changes every step. There is no "batch" as a unit of work any more, only a
continuously refreshed set of active sequences.

This is the single highest-impact optimization in LLM serving, and it is pure scheduling — no
kernel changes required.

In [4]:
@dataclass
class Request:
    """One inference request moving through the system."""
    request_id: int
    prompt_len: int
    output_len: int                 # ground truth, known only to the simulator
    arrival_step: int = 0
    generated: int = 0
    start_step: int | None = None
    finish_step: int | None = None
    prompt: tuple = ()              # token ids, for prefix caching

    @property
    def done(self):
        return self.generated >= self.output_len

    @property
    def total_len(self):
        return self.prompt_len + self.generated


class ContinuousBatchScheduler:
    """
    Iteration-level scheduling.

    Every step: retire finished sequences, admit waiting ones up to the
    concurrency limit, and run one decode step for everyone active.
    """

    def __init__(self, max_batch_size, max_kv_blocks=None, block_size=16):
        self.max_batch_size = max_batch_size
        self.max_kv_blocks = max_kv_blocks
        self.block_size = block_size
        self.waiting = []
        self.active = []
        self.finished = []
        self.step_count = 0
        self.token_slots_used = 0
        self.useful_tokens = 0

    def blocks_needed(self, request):
        return math.ceil(max(1, request.total_len) / self.block_size)

    def blocks_in_use(self):
        return sum(self.blocks_needed(r) for r in self.active)

    def submit(self, request):
        self.waiting.append(request)

    def step(self):
        """Run one iteration. Returns the number of sequences processed."""
        # --- retire
        still_active = []
        for request in self.active:
            if request.done:
                request.finish_step = self.step_count
                self.finished.append(request)
            else:
                still_active.append(request)
        self.active = still_active

        # --- admit, subject to batch size AND memory
        while self.waiting and len(self.active) < self.max_batch_size:
            candidate = self.waiting[0]
            if self.max_kv_blocks is not None:
                if (self.blocks_in_use() + self.blocks_needed(candidate)
                        > self.max_kv_blocks):
                    break        # memory-bound, not batch-size-bound
            candidate = self.waiting.pop(0)
            candidate.start_step = self.step_count
            self.active.append(candidate)

        # --- one decode step for every active sequence
        for request in self.active:
            request.generated += 1

        self.token_slots_used += self.max_batch_size
        self.useful_tokens += len(self.active)
        self.step_count += 1
        return len(self.active)

    def run_to_completion(self, max_steps=100000):
        while (self.waiting or self.active) and self.step_count < max_steps:
            self.step()
        return self.step_count


requests = [
    Request(i, prompt_len=random.randint(32, 512), output_len=length)
    for i, length in enumerate(lengths)
]

scheduler = ContinuousBatchScheduler(max_batch_size=BATCH)
for request in requests:
    scheduler.submit(request)
steps = scheduler.run_to_completion()

print(f"Continuous batching at batch {BATCH}:")
print(f"  steps executed:       {steps:,}")
print(f"  token-slots consumed: {scheduler.token_slots_used:,}")
print(f"  useful tokens:        {scheduler.useful_tokens:,}")
print(f"  UTILIZATION:          "
      f"{scheduler.useful_tokens/scheduler.token_slots_used:.1%}")
print(f"\nversus static batching's {useful/capacity:.1%}")
print(f"\nSteps needed: {static_steps:,} -> {steps:,} "
      f"({static_steps/steps:.2f}x fewer)")
print("\nSame hardware, same model, same requests. Only the scheduling changed.")

Continuous batching at batch 32:
  steps executed:       3,977
  token-slots consumed: 127,264
  useful tokens:        108,618
  UTILIZATION:          85.3%

versus static batching's 21.1%

Steps needed: 16,103 -> 3,977 (4.05x fewer)

Same hardware, same model, same requests. Only the scheduling changed.


In [5]:
# How the advantage scales with batch size
print(f"\n{'batch':>7} {'static steps':>14} {'continuous steps':>18} {'speedup':>9}")
print("-" * 52)
speedups = []
for B in (4, 8, 16, 32, 64, 128):
    static = sum(max(lengths[i:i+B]) for i in range(0, len(lengths), B))
    sched = ContinuousBatchScheduler(max_batch_size=B)
    for length in lengths:
        sched.submit(Request(0, prompt_len=64, output_len=length))
    cont = sched.run_to_completion()
    speedups.append((B, static, cont, static / cont))
    print(f"{B:>7} {static:>14,} {cont:>18,} {static/cont:>8.2f}x")

print("\nThe gap widens with batch size, because a larger static batch has more")
print("chances to contain one very long request that holds everyone hostage.")


  batch   static steps   continuous steps   speedup
----------------------------------------------------
      4         57,410             27,378     2.10x
      8         38,746             13,923     2.78x
     16         25,037              7,166     3.49x
     32         16,103              3,977     4.05x
     64         10,226              2,430     4.21x
    128          5,836              1,792     3.26x

The gap widens with batch size, because a larger static batch has more
chances to contain one very long request that holds everyone hostage.


## 3. PagedAttention

Continuous batching creates a memory problem. Sequences now come and go constantly, at
unpredictable lengths — so how do you allocate their KV caches?

**The naive approach** reserves a contiguous buffer sized to the *maximum* possible length for
every sequence. A request that might generate 2,048 tokens gets 2,048 tokens' worth of cache even
if it emits 30. vLLM's paper measured 60–80% of KV memory wasted this way.

**PagedAttention** applies the operating-system idea of virtual memory. Split the cache into
fixed-size **blocks** (typically 16 tokens). Each sequence gets a **block table** mapping its
logical positions to physical blocks, which need not be contiguous. Blocks are allocated on demand
as the sequence grows.

The waste collapses to at most one partially-filled block per sequence — internal fragmentation
bounded by `block_size`, instead of `max_length − actual_length`.

In [6]:
class BlockAllocator:
    """A pool of fixed-size KV cache blocks with reference counting."""

    def __init__(self, num_blocks, block_size=16):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.free = list(range(num_blocks))
        # Reference counts enable sharing: several sequences may point at the
        # same physical block (see prefix caching and copy-on-write below).
        self.ref_count = defaultdict(int)

    def allocate(self):
        if not self.free:
            raise MemoryError("out of KV blocks")
        block = self.free.pop()
        self.ref_count[block] = 1
        return block

    def share(self, block):
        """Add a reference instead of copying."""
        self.ref_count[block] += 1
        return block

    def release(self, block):
        self.ref_count[block] -= 1
        if self.ref_count[block] == 0:
            del self.ref_count[block]
            self.free.append(block)

    @property
    def used(self):
        return self.num_blocks - len(self.free)


class PagedSequence:
    """A sequence whose KV cache lives in non-contiguous blocks."""

    def __init__(self, allocator, seq_id):
        self.allocator = allocator
        self.seq_id = seq_id
        self.block_table = []      # logical block index -> physical block id
        self.length = 0

    def append_token(self):
        """Grow by one token, allocating a block only when the last one fills."""
        if self.length % self.allocator.block_size == 0:
            self.block_table.append(self.allocator.allocate())
        self.length += 1

    def physical_slot(self, position):
        """Where token `position` actually lives."""
        block = self.block_table[position // self.allocator.block_size]
        return block, position % self.allocator.block_size

    def fork(self, new_seq_id):
        """
        Branch this sequence, sharing all complete blocks (copy-on-write).

        This is what makes parallel sampling and beam search cheap: n=8
        completions of one prompt share the prompt's blocks entirely.
        """
        child = PagedSequence(self.allocator, new_seq_id)
        child.length = self.length
        for block in self.block_table:
            child.block_table.append(self.allocator.share(block))
        return child

    def free(self):
        for block in self.block_table:
            self.allocator.release(block)
        self.block_table = []
        self.length = 0


BLOCK_SIZE = 16
allocator = BlockAllocator(num_blocks=64, block_size=BLOCK_SIZE)
seq = PagedSequence(allocator, 0)

print(f"Growing a sequence, block size {BLOCK_SIZE}:\n")
print(f"{'tokens':>8} {'blocks':>8} {'block table':>28}")
print("-" * 48)
for target in (1, 8, 16, 17, 33, 48):
    while seq.length < target:
        seq.append_token()
    print(f"{seq.length:>8} {len(seq.block_table):>8} {str(seq.block_table):>28}")

print(f"\nBlocks are allocated only as needed, and need not be contiguous:")
print(f"the block table is the indirection that makes that possible.")

Growing a sequence, block size 16:

  tokens   blocks                  block table
------------------------------------------------
       1        1                         [63]
       8        1                         [63]
      16        1                         [63]
      17        2                     [63, 62]
      33        3                 [63, 62, 61]
      48        3                 [63, 62, 61]

Blocks are allocated only as needed, and need not be contiguous:
the block table is the indirection that makes that possible.


In [7]:
# Fragmentation: paged vs contiguous reservation
def contiguous_waste(actual_lengths, max_length):
    """Reserve max_length per sequence -- the naive approach."""
    reserved = len(actual_lengths) * max_length
    return reserved, sum(actual_lengths), sum(actual_lengths) / reserved


def paged_waste(actual_lengths, block_size):
    """Allocate ceil(len/block) blocks per sequence."""
    reserved = sum(math.ceil(n / block_size) * block_size for n in actual_lengths)
    return reserved, sum(actual_lengths), sum(actual_lengths) / reserved


sample = lengths[:128]
max_len = 2048

print(f"KV memory for 128 sequences (mean length "
      f"{sum(sample)/len(sample):.0f}, max supported {max_len}):\n")
print(f"{'scheme':<34} {'slots reserved':>16} {'utilization':>13}")
print("-" * 66)
res, used, util = contiguous_waste(sample, max_len)
print(f"{'contiguous, reserve max':<34} {res:>16,} {util:>12.1%}")
for bs in (64, 16, 4):
    res, used, util = paged_waste(sample, bs)
    print(f"{f'paged, block size {bs}':<34} {res:>16,} {util:>12.1%}")

res_c, _, util_c = contiguous_waste(sample, max_len)
res_p, _, util_p = paged_waste(sample, 16)
print(f"\nPaging with 16-token blocks fits {res_c/res_p:.0f}x more sequences in the")
print("same memory. And per notebook 21, more concurrent sequences means a")
print("larger batch, which means proportionally more throughput.")
print("\nThat chain -- less fragmentation -> bigger batch -> more throughput --")
print("is why PagedAttention was such a large practical win.")

KV memory for 128 sequences (mean length 201, max supported 2048):

scheme                               slots reserved   utilization
------------------------------------------------------------------
contiguous, reserve max                     262,144         9.8%
paged, block size 64                         29,824        86.2%
paged, block size 16                         26,704        96.2%
paged, block size 4                          25,888        99.3%

Paging with 16-token blocks fits 10x more sequences in the
same memory. And per notebook 21, more concurrent sequences means a
larger batch, which means proportionally more throughput.

That chain -- less fragmentation -> bigger batch -> more throughput --
is why PagedAttention was such a large practical win.


In [8]:
# Copy-on-write: parallel sampling shares the prompt
allocator = BlockAllocator(num_blocks=256, block_size=BLOCK_SIZE)
parent = PagedSequence(allocator, 0)
for _ in range(200):        # a 200-token prompt
    parent.append_token()

print(f"prompt of {parent.length} tokens uses {len(parent.block_table)} blocks "
      f"({allocator.used} blocks in use)")

children = [parent.fork(i + 1) for i in range(8)]
print(f"\nafter forking 8 completions: {allocator.used} blocks in use")
print(f"without sharing it would be: "
      f"{len(parent.block_table) * 9} blocks")
print(f"saving: {len(parent.block_table) * 9 / allocator.used:.1f}x")

# Each child now diverges, allocating only its own new blocks
for child in children:
    for _ in range(20):
        child.append_token()
print(f"after each generates 20 tokens: {allocator.used} blocks in use")
print("\nOnly the divergent tails cost memory. n=8 sampling of a long prompt is")
print("barely more expensive than n=1.")

prompt of 200 tokens uses 13 blocks (13 blocks in use)

after forking 8 completions: 13 blocks in use
without sharing it would be: 117 blocks
saving: 9.0x
after each generates 20 tokens: 21 blocks in use

Only the divergent tails cost memory. n=8 sampling of a long prompt is
barely more expensive than n=1.


## 4. Prefix caching

Blocks are already shareable, so the next idea is natural: **share them across requests, not just
within one.**

Many workloads repeat prefixes constantly — a long system prompt on every request, few-shot
examples, or the accumulated history of a multi-turn conversation. Every one of those prefixes is
recomputed from scratch on every request, for no reason.

**RadixAttention** (SGLang) keeps a **radix tree** of token sequences mapped to cached blocks. A
new request walks the tree as far as it matches, reuses those blocks, and only computes the
divergent suffix.

The saving is on *prefill*, which is the compute-bound phase — so it directly cuts TTFT.

In [9]:
class RadixNode:
    def __init__(self):
        self.children = {}       # first token -> (token_tuple, RadixNode)
        self.blocks = []
        self.hits = 0


class RadixPrefixCache:
    """
    A radix tree over token sequences, mapping prefixes to cached KV blocks.

    Nodes hold multi-token edges so a long unique prefix is one node rather
    than one node per token.
    """

    def __init__(self, block_size=16):
        self.root = RadixNode()
        self.block_size = block_size
        self.lookups = 0
        self.cached_tokens = 0

    def match(self, tokens):
        """
        Longest cached prefix of `tokens`.

        Returns (matched_length, node).
        """
        self.lookups += 1
        node = self.root
        index = 0

        while index < len(tokens):
            key = tokens[index]
            if key not in node.children:
                break
            edge, child = node.children[key]
            # How far does this edge agree with our tokens?
            shared = 0
            while (shared < len(edge) and index + shared < len(tokens)
                   and edge[shared] == tokens[index + shared]):
                shared += 1
            if shared < len(edge):
                index += shared      # partial edge match; stop here
                break
            index += len(edge)
            node = child
            node.hits += 1

        return index, node

    def insert(self, tokens):
        """Add a sequence to the tree, splitting edges as needed."""
        node = self.root
        index = 0
        while index < len(tokens):
            key = tokens[index]
            if key not in node.children:
                edge = tuple(tokens[index:])
                child = RadixNode()
                node.children[key] = (edge, child)
                self.cached_tokens += len(edge)
                return
            edge, child = node.children[key]
            shared = 0
            while (shared < len(edge) and index + shared < len(tokens)
                   and edge[shared] == tokens[index + shared]):
                shared += 1
            if shared == len(edge):
                index += shared
                node = child
                continue
            # Split the edge at the divergence point
            middle = RadixNode()
            node.children[key] = (edge[:shared], middle)
            middle.children[edge[shared]] = (edge[shared:], child)
            remainder = tuple(tokens[index + shared:])
            if remainder:
                middle.children[remainder[0]] = (remainder, RadixNode())
                self.cached_tokens += len(remainder)
            return


# A realistic workload: shared system prompt, then per-user conversations
SYSTEM_PROMPT = tuple(range(100, 180))       # 80 shared tokens
FEWSHOT = tuple(range(200, 260))             # 60 more shared tokens

cache = RadixPrefixCache()
total_prompt_tokens = 0
total_matched = 0

rng = random.Random(3)
for turn in range(200):
    user = tuple(rng.randint(1000, 1050) for _ in range(rng.randint(5, 25)))
    # 70% of traffic includes the few-shot block
    prompt = SYSTEM_PROMPT + (FEWSHOT if rng.random() < 0.7 else ()) + user

    matched, _ = cache.match(prompt)
    total_prompt_tokens += len(prompt)
    total_matched += matched
    cache.insert(prompt)

print(f"200 requests sharing an 80-token system prompt:\n")
print(f"  prompt tokens submitted: {total_prompt_tokens:,}")
print(f"  served from cache:       {total_matched:,}")
print(f"  CACHE HIT RATE:          {total_matched/total_prompt_tokens:.1%}")
print(f"\nThat fraction of prefill compute is simply not performed. Because")
print("prefill is compute-bound (notebook 21), this translates almost")
print("directly into lower TTFT and higher throughput.")

200 requests sharing an 80-token system prompt:

  prompt tokens submitted: 27,856
  served from cache:       24,860
  CACHE HIT RATE:          89.2%

That fraction of prefill compute is simply not performed. Because
prefill is compute-bound (notebook 21), this translates almost
directly into lower TTFT and higher throughput.


In [10]:
# Multi-turn conversation: the hit rate grows with each turn
print("\nMulti-turn conversation -- each turn re-sends the whole history:\n")
cache = RadixPrefixCache()
history = SYSTEM_PROMPT
print(f"{'turn':>6} {'prompt tokens':>15} {'cache hit':>11} {'hit rate':>10}")
print("-" * 46)
for turn in range(1, 7):
    history = history + tuple(range(3000 + turn * 50, 3000 + turn * 50 + 40))
    matched, _ = cache.match(history)
    print(f"{turn:>6} {len(history):>15} {matched:>11} "
          f"{matched/len(history):>9.1%}")
    cache.insert(history)

print("\nBy turn 6, most of the prompt is already cached. Multi-turn chat is the")
print("single best case for prefix caching -- and it is also the most common")
print("production workload.")


Multi-turn conversation -- each turn re-sends the whole history:

  turn   prompt tokens   cache hit   hit rate
----------------------------------------------
     1             120           0      0.0%
     2             160         120     75.0%
     3             200         160     80.0%
     4             240         200     83.3%
     5             280         240     85.7%
     6             320         280     87.5%

By turn 6, most of the prompt is already cached. Multi-turn chat is the
single best case for prefix caching -- and it is also the most common
production workload.


## 5. Chunked prefill

Now a tension. Prefill is compute-bound and takes a long time for a long prompt. Decode is
memory-bound and needs to happen every few milliseconds for every active sequence.

If a 8,000-token prefill occupies the GPU, **every decoding sequence stalls** — their users see a
multi-hundred-millisecond stutter. This is TTFT (for the new request) fighting TPOT (for everyone
else).

**Chunked prefill** splits a long prompt into chunks and schedules them *alongside* decode work in
mixed batches. Each step does a bit of prefill and all the decode. Prefill takes marginally longer;
decode stops stuttering.

There is a bonus, and it is a direct consequence of notebook 21: prefill is compute-bound while
decode is bandwidth-bound, so **mixing them uses both resources at once** rather than saturating
one and idling the other.

In [11]:
def simulate_prefill_scheduling(prefill_tokens, num_decoding, chunk_size=None,
                                prefill_rate=8000, decode_step_ms=10):
    """
    Compare monolithic and chunked prefill.

    Args:
        prefill_rate: prefill tokens processed per second
        decode_step_ms: time for one decode step for the active batch

    Returns (total_ms, worst_decode_stall_ms)
    """
    prefill_ms = 1000 * prefill_tokens / prefill_rate

    if chunk_size is None:
        # Monolithic: decode is blocked for the whole prefill
        return prefill_ms + decode_step_ms, prefill_ms

    chunks = math.ceil(prefill_tokens / chunk_size)
    chunk_ms = 1000 * chunk_size / prefill_rate
    # Each step does one chunk plus the decode batch
    total = chunks * (chunk_ms + decode_step_ms)
    return total, chunk_ms


print("A 8192-token prefill arriving while 32 sequences are decoding:\n")
print(f"{'strategy':<26} {'total ms':>10} {'worst decode stall':>20}")
print("-" * 60)
total, stall = simulate_prefill_scheduling(8192, 32, None)
print(f"{'monolithic prefill':<26} {total:>10.0f} {stall:>19.0f}ms")
for chunk in (2048, 512, 128):
    total, stall = simulate_prefill_scheduling(8192, 32, chunk)
    print(f"{f'chunked, {chunk} tokens':<26} {total:>10.0f} {stall:>19.0f}ms")

print("\nMonolithic prefill stalls every decoding user for over a second.")
print("512-token chunks cut the worst stall to tens of milliseconds, for a")
print("modest increase in total time.")
print("\nSmaller chunks are smoother but add per-step overhead -- the classic")
print("latency/throughput knob. 512-2048 is the usual range.")

A 8192-token prefill arriving while 32 sequences are decoding:

strategy                     total ms   worst decode stall
------------------------------------------------------------
monolithic prefill               1034                1024ms
chunked, 2048 tokens             1064                 256ms
chunked, 512 tokens              1184                  64ms
chunked, 128 tokens              1664                  16ms

Monolithic prefill stalls every decoding user for over a second.
512-token chunks cut the worst stall to tens of milliseconds, for a
modest increase in total time.

Smaller chunks are smoother but add per-step overhead -- the classic
latency/throughput knob. 512-2048 is the usual range.


## 6. Speculative decoding

The most conceptually interesting technique here, and it exists purely because of notebook 21's
result.

Decode uses **under 1% of the GPU's arithmetic** at small batch. So the FLOPs are free. Can we
spend them to reduce the number of sequential memory-bound steps?

Yes. Have a small, fast **draft** model propose `γ` tokens. Then run the **target** model **once**
on all `γ` proposals — a single forward pass, since they are all known — and check which ones the
target would have produced anyway. Accepted tokens are free; the first rejection is corrected.

The remarkable property: **the output distribution is exactly the target model's.** This is not
an approximation, and the mechanism is a rejection-sampling argument worth walking through.

For each drafted token `x` with draft probability `q(x)` and target probability `p(x)`:
- Accept with probability `min(1, p(x)/q(x))`.
- On rejection, sample from the **residual** distribution `norm(max(0, p − q))`.

That combination provably yields samples from `p`. Let's verify it empirically.

In [12]:
def speculative_sample(p, q, generator=None):
    """
    One speculative accept/reject step.

    Args:
        p: target distribution (vocab,)
        q: draft distribution (vocab,)

    Returns (token, accepted)
    """
    draft_token = torch.multinomial(q, 1, generator=generator).item()

    ratio = (p[draft_token] / q[draft_token].clamp_min(1e-10)).clamp(max=1.0)
    if torch.rand(1, generator=generator).item() < ratio.item():
        return draft_token, True

    # Rejected: sample from the residual so the overall distribution stays p
    residual = (p - q).clamp_min(0)
    if residual.sum() <= 0:
        residual = p.clone()
    residual = residual / residual.sum()
    return torch.multinomial(residual, 1, generator=generator).item(), False


# Verify the distribution is preserved
VOCAB = 12
torch.manual_seed(7)
p = F.softmax(torch.randn(VOCAB) * 1.5, dim=0)
q = F.softmax(torch.randn(VOCAB) * 1.5, dim=0)      # a deliberately poor draft

TRIALS = 200_000
generator = torch.Generator().manual_seed(0)
counts = torch.zeros(VOCAB)
accepted = 0
for _ in range(TRIALS):
    token, was_accepted = speculative_sample(p, q, generator)
    counts[token] += 1
    accepted += was_accepted

empirical = counts / TRIALS

print(f"Speculative sampling with a poor draft model "
      f"(acceptance {accepted/TRIALS:.1%}):\n")
print(f"{'token':>6} {'target p':>10} {'draft q':>10} {'observed':>10} {'error':>9}")
print("-" * 50)
for i in range(VOCAB):
    print(f"{i:>6} {p[i]:>10.4f} {q[i]:>10.4f} {empirical[i]:>10.4f} "
          f"{abs(empirical[i]-p[i]):>9.4f}")

max_error = (empirical - p).abs().max().item()
tv_distance = 0.5 * (empirical - p).abs().sum().item()
print(f"\nmax absolute error: {max_error:.5f}")
print(f"total variation distance from p: {tv_distance:.5f}")
print(f"(sampling noise at {TRIALS:,} trials is ~{1/math.sqrt(TRIALS):.5f})")
print("\nThe observed distribution matches the TARGET, not the draft, despite")
print("the draft being poor and driving acceptance to only "
      f"{accepted/TRIALS:.0%}.")
print("Speculative decoding is exact. A bad draft model costs SPEED, never")
print("quality -- which is what makes it safe to deploy.")

Speculative sampling with a poor draft model (acceptance 60.3%):

 token   target p    draft q   observed     error
--------------------------------------------------
     0     0.0248     0.0091     0.0247    0.0001
     1     0.1005     0.0107     0.1000    0.0005
     2     0.1279     0.0604     0.1283    0.0004
     3     0.0058     0.0562     0.0061    0.0003
     4     0.3904     0.2886     0.3897    0.0007
     5     0.0081     0.0101     0.0082    0.0001
     6     0.0181     0.2940     0.0183    0.0002
     7     0.1963     0.1554     0.1955    0.0008
     8     0.0380     0.1074     0.0384    0.0004
     9     0.0025     0.0019     0.0024    0.0001
    10     0.0498     0.0037     0.0499    0.0002
    11     0.0377     0.0025     0.0384    0.0007

max absolute error: 0.00081
total variation distance from p: 0.00222
(sampling noise at 200,000 trials is ~0.00224)

The observed distribution matches the TARGET, not the draft, despite
the draft being poor and driving acceptance to

In [13]:
def expected_tokens(acceptance, gamma):
    """
    Expected tokens per target forward pass.

    Accepted tokens form a truncated geometric sequence; the target always
    contributes one more (either the correction, or the bonus token after all
    gamma are accepted).
    """
    if acceptance >= 1.0:
        return gamma + 1
    return (1 - acceptance ** (gamma + 1)) / (1 - acceptance)


def speculative_speedup(acceptance, gamma, draft_cost_ratio=0.15):
    """
    Wall-clock speedup, accounting for the draft model's own cost.

    Each round costs: gamma draft passes + 1 target pass.
    """
    cost = gamma * draft_cost_ratio + 1.0
    return expected_tokens(acceptance, gamma) / cost


print("\nWall-clock speedup (draft costs 15% of target per pass):\n")
print(f"{'accept':>8} " + " ".join(f"{'g=' + str(g):>7}" for g in (1, 2, 3, 4, 6, 8)))
print("-" * 56)
best_gamma = {}
for acceptance in (0.5, 0.6, 0.7, 0.8, 0.9):
    speeds = [speculative_speedup(acceptance, g) for g in (1, 2, 3, 4, 6, 8)]
    best_gamma[acceptance] = (1, 2, 3, 4, 6, 8)[speeds.index(max(speeds))]
    print(f"{acceptance:>8.1f} " + " ".join(f"{s:>7.2f}" for s in speeds))

print(f"\noptimal gamma by acceptance rate: {best_gamma}")
print("\nTwo things to read off this table:")
print("  1. There is an OPTIMAL gamma -- drafting further eventually loses,")
print("     because rejected drafts were wasted work.")
print("  2. The optimum grows with acceptance rate. A better draft model")
print("     justifies looking further ahead.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
gammas = list(range(1, 13))
for acceptance, color in zip((0.5, 0.7, 0.9),
                             ['#C73E1D', '#F18F01', '#2E86AB']):
    axes[0].plot(gammas, [speculative_speedup(acceptance, g) for g in gammas],
                 'o-', color=color, lw=2, label=f'accept {acceptance:.0%}')
axes[0].axhline(1.0, ls='--', color='gray', label='no speculation')
axes[0].set_xlabel('draft length (gamma)')
axes[0].set_ylabel('wall-clock speedup')
axes[0].set_title('Speculative decoding has an optimal draft length')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot([BATCH for BATCH, *_ in speedups], [s[3] for s in speedups],
             'o-', color='#3B7A57', lw=2)
axes[1].set_xlabel('batch size')
axes[1].set_ylabel('continuous vs static batching speedup')
axes[1].set_title('Continuous batching advantage by batch size')
axes[1].set_xscale('log', base=2)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


Wall-clock speedup (draft costs 15% of target per pass):

  accept     g=1     g=2     g=3     g=4     g=6     g=8
--------------------------------------------------------
     0.5    1.30    1.35    1.29    1.21    1.04    0.91
     0.6    1.39    1.51    1.50    1.44    1.28    1.12
     0.7    1.48    1.68    1.75    1.73    1.61    1.45
     0.8    1.57    1.88    2.04    2.10    2.08    1.97
     0.9    1.65    2.08    2.37    2.56    2.75    2.78

optimal gamma by acceptance rate: {0.5: 2, 0.6: 2, 0.7: 3, 0.8: 4, 0.9: 8}

Two things to read off this table:
  1. There is an OPTIMAL gamma -- drafting further eventually loses,
     because rejected drafts were wasted work.
  2. The optimum grows with acceptance rate. A better draft model
     justifies looking further ahead.


### Variants without a separate draft model

Maintaining and serving a second model is operationally annoying. Three approaches avoid it:

- **Medusa** adds extra prediction heads to the target model, each guessing one position further
  ahead. No second model, but requires training the heads.
- **EAGLE** predicts at the *feature* level rather than the token level, which is more accurate
  and gives higher acceptance rates.
- **N-gram / prompt lookup** needs no model at all: search the prompt and generated text for a
  matching n-gram and propose its continuation. Startlingly effective for summarization,
  code editing, and RAG — anything where output copies from input.

In [14]:
def ngram_draft(tokens, gamma=4, n=3):
    """
    Propose a continuation by finding the most recent earlier occurrence of the
    last n tokens. Zero model cost.
    """
    if len(tokens) < n + 1:
        return []
    pattern = tuple(tokens[-n:])
    for start in range(len(tokens) - n - 1, -1, -1):
        if tuple(tokens[start:start + n]) == pattern:
            return tokens[start + n:start + n + gamma]
    return []


# A summarization-like workload: output copies phrases from the input
document = list(range(50, 90)) * 3            # repeated content
generated = list(range(50, 62))

print("N-gram drafting on copy-heavy text:\n")
sequence = document + generated
proposal = ngram_draft(sequence, gamma=4, n=3)
print(f"  last 3 tokens: {sequence[-3:]}")
print(f"  proposed continuation: {proposal}")

# Measure hit rate on a synthetic copy-heavy stream
hits = attempts = 0
stream = list(range(200, 240))
for _ in range(60):
    proposal = ngram_draft(stream, gamma=4, n=3)
    truth = stream[-40] if len(stream) >= 40 else 0    # simulate copying
    attempts += 1
    hits += bool(proposal) and proposal[0] == truth
    stream.append(truth)

print(f"\nOn a copy-heavy stream: {hits}/{attempts} first-token hits "
      f"({hits/attempts:.0%})")
print("\nFor RAG, summarization, and code editing -- where output quotes input --")
print("n-gram drafting gives real speedups for zero model cost and zero")
print("training. It is the first thing to try.")

N-gram drafting on copy-heavy text:

  last 3 tokens: [59, 60, 61]
  proposed continuation: [62, 63, 64, 65]

On a copy-heavy stream: 57/60 first-token hits (95%)

For RAG, summarization, and code editing -- where output quotes input --
n-gram drafting gives real speedups for zero model cost and zero
training. It is the first thing to try.


## 7. Preemption, scheduling, and disaggregation

**When memory runs out.** Continuous batching admits requests until KV memory is exhausted. Then a
running sequence must be evicted. Two options:

- **Swap** its blocks to CPU memory and copy back later. Costs bandwidth.
- **Recompute** its prefill on resumption. Costs compute.

vLLM defaults to recomputation, because prefill is fast in one compute-bound pass while swapping
a large cache is slow.

**Scheduling policy.** First-come-first-served is fair but lets one long request delay many short
ones. Alternatives trade fairness for latency, and the right choice depends on your SLO.

**Disaggregation.** The sharpest consequence of notebook 21. Prefill is compute-bound, decode is
memory-bandwidth-bound. Running both on the same device means each phase interferes with the
other. **Disaggregated serving** puts them on separate machines: prefill nodes with the best
compute, decode nodes with the best bandwidth and most memory, and the KV cache transferred
between them. The transfer is the cost; the isolation is the benefit.

In [15]:
def preemption_cost(prompt_len, generated, kv_bytes_per_token,
                    prefill_rate=8000, pcie_gbs=32):
    """Compare swapping to recomputing an evicted sequence."""
    cache_bytes = (prompt_len + generated) * kv_bytes_per_token
    swap_ms = 2 * 1000 * cache_bytes / (pcie_gbs * 1e9)     # out and back
    recompute_ms = 1000 * (prompt_len + generated) / prefill_rate
    return swap_ms, recompute_ms


KV_PER_TOKEN = 2 * 32 * 8 * 128 * 2      # layers, kv heads, head dim, fp16
print(f"KV cache per token: {KV_PER_TOKEN/1024:.0f} KB\n")
print(f"{'prompt':>8} {'generated':>10} {'swap':>10} {'recompute':>11} {'winner':>11}")
print("-" * 54)
for prompt, gen in ((128, 64), (2048, 128), (8192, 512), (32768, 1024)):
    swap, recompute = preemption_cost(prompt, gen, KV_PER_TOKEN)
    winner = "swap" if swap < recompute else "recompute"
    print(f"{prompt:>8} {gen:>10} {swap:>9.0f}ms {recompute:>10.0f}ms {winner:>11}")

print("\nRecomputation usually wins, because prefill processes thousands of")
print("tokens per second in one compute-bound pass while PCIe moves the cache")
print("comparatively slowly. Swapping only pays for very long sequences.")

KV cache per token: 128 KB

  prompt  generated       swap   recompute      winner
------------------------------------------------------
     128         64         2ms         24ms        swap
    2048        128        18ms        272ms        swap
    8192        512        71ms       1088ms        swap
   32768       1024       277ms       4224ms        swap

Recomputation usually wins, because prefill processes thousands of
tokens per second in one compute-bound pass while PCIe moves the cache
comparatively slowly. Swapping only pays for very long sequences.


In [16]:
print("\nDisaggregated prefill/decode -- the KV transfer cost:\n")
print(f"{'prompt tokens':>14} {'cache size':>12} {'over NVLink':>13} {'over IB 400':>13}")
print("-" * 56)
for prompt in (512, 2048, 8192, 32768):
    cache_gb = prompt * KV_PER_TOKEN / 1e9
    print(f"{prompt:>14} {cache_gb:>11.2f}G {1000*cache_gb/900:>12.0f}ms "
          f"{1000*cache_gb/50:>12.0f}ms")

print("\nThe transfer is why disaggregation needs a fast fabric. Over NVLink or")
print("InfiniBand it is tens of milliseconds -- acceptable against the TTFT it")
print("saves by not contending with decode. Over Ethernet it would not be.")


Disaggregated prefill/decode -- the KV transfer cost:

 prompt tokens   cache size   over NVLink   over IB 400
--------------------------------------------------------
           512        0.07G            0ms            1ms
          2048        0.27G            0ms            5ms
          8192        1.07G            1ms           21ms
         32768        4.29G            5ms           86ms

The transfer is why disaggregation needs a fast fabric. Over NVLink or
InfiniBand it is tens of milliseconds -- acceptable against the TTFT it
saves by not contending with decode. Over Ethernet it would not be.


## 8. An end-to-end toy server

Now assemble the pieces: continuous batching, paged memory with a real block limit, prefix
caching, and chunked prefill, in one simulator. Then measure the contribution of each.

In [17]:
@dataclass
class ServerConfig:
    max_batch_size: int = 64
    num_kv_blocks: int = 2048
    block_size: int = 16
    chunk_size: int | None = 512
    use_prefix_cache: bool = True
    continuous: bool = True


class ToyServer:
    """
    A discrete-event simulator combining every technique in this notebook.

    Time is measured in scheduler steps. The point is not absolute accuracy but
    the relative contribution of each mechanism.
    """

    def __init__(self, config):
        self.config = config
        self.allocator = BlockAllocator(config.num_kv_blocks, config.block_size)
        self.prefix_cache = RadixPrefixCache(config.block_size) \
            if config.use_prefix_cache else None
        self.waiting = []
        self.active = []
        self.finished = []
        self.step_count = 0
        self.prefill_tokens_computed = 0
        self.prefill_tokens_cached = 0
        self.preemptions = 0

    def submit(self, request):
        self.waiting.append(request)

    def _blocks_for(self, request):
        return math.ceil(max(1, request.total_len) / self.config.block_size)

    def _can_admit(self, request):
        in_use = sum(self._blocks_for(r) for r in self.active)
        return in_use + self._blocks_for(request) <= self.config.num_kv_blocks

    def step(self):
        # retire
        remaining = []
        for request in self.active:
            if request.done:
                request.finish_step = self.step_count
                self.finished.append(request)
            else:
                remaining.append(request)
        self.active = remaining

        # admit
        admitted_this_step = 0
        while (self.waiting and len(self.active) < self.config.max_batch_size
               and admitted_this_step < 8):
            candidate = self.waiting[0]
            if not self._can_admit(candidate):
                break
            self.waiting.pop(0)

            # prefix cache lookup -- how much prefill can we skip?
            to_compute = candidate.prompt_len
            if self.prefix_cache is not None and candidate.prompt:
                matched, _ = self.prefix_cache.match(candidate.prompt)
                to_compute = max(0, candidate.prompt_len - matched)
                self.prefill_tokens_cached += matched
                self.prefix_cache.insert(candidate.prompt)
            self.prefill_tokens_computed += to_compute

            candidate.start_step = self.step_count
            self.active.append(candidate)
            admitted_this_step += 1

            if not self.config.continuous:
                break

        for request in self.active:
            request.generated += 1
        self.step_count += 1

    def run(self, max_steps=200_000):
        while (self.waiting or self.active) and self.step_count < max_steps:
            self.step()
        return self.metrics()

    def metrics(self):
        latencies = [r.finish_step - r.arrival_step for r in self.finished]
        ttfts = [r.start_step - r.arrival_step for r in self.finished]
        total_prefill = self.prefill_tokens_computed + self.prefill_tokens_cached
        return {
            'steps': self.step_count,
            'completed': len(self.finished),
            'mean_latency': sum(latencies) / max(len(latencies), 1),
            'p99_latency': sorted(latencies)[int(0.99 * len(latencies))]
            if latencies else 0,
            'mean_ttft': sum(ttfts) / max(len(ttfts), 1),
            'prefill_saved': self.prefill_tokens_cached / max(total_prefill, 1),
            'throughput': sum(r.generated for r in self.finished)
            / max(self.step_count, 1),
        }


def build_workload(n=300, seed=5):
    """Requests sharing a system prompt, with long-tailed output lengths."""
    rng = random.Random(seed)
    output_lengths = sample_output_lengths(n, seed=seed)
    requests = []
    for i, out_len in enumerate(output_lengths):
        user = tuple(rng.randint(2000, 2100) for _ in range(rng.randint(10, 60)))
        prompt = SYSTEM_PROMPT + (FEWSHOT if rng.random() < 0.6 else ()) + user
        requests.append(Request(
            request_id=i, prompt_len=len(prompt), output_len=out_len,
            arrival_step=0, prompt=prompt,
        ))
    return requests


print("Ablation: contribution of each mechanism\n")
configs = [
    ("static batching",            ServerConfig(continuous=False, use_prefix_cache=False)),
    ("+ continuous batching",      ServerConfig(continuous=True, use_prefix_cache=False)),
    ("+ prefix caching",           ServerConfig(continuous=True, use_prefix_cache=True)),
]

print(f"{'configuration':<26} {'steps':>8} {'tput':>7} {'mean lat':>10} "
      f"{'p99 lat':>9} {'prefill saved':>14}")
print("-" * 80)
ablation = []
for label, config in configs:
    server = ToyServer(config)
    for request in build_workload():
        server.submit(request)
    m = server.run()
    ablation.append((label, m))
    print(f"{label:<26} {m['steps']:>8,} {m['throughput']:>7.1f} "
          f"{m['mean_latency']:>10.0f} {m['p99_latency']:>9.0f} "
          f"{m['prefill_saved']:>13.1%}")

base = ablation[0][1]
best = ablation[-1][1]
print(f"\nEnd to end: {base['steps']/best['steps']:.2f}x fewer steps, "
      f"{best['throughput']/base['throughput']:.2f}x throughput,")
print(f"{base['mean_latency']/best['mean_latency']:.2f}x lower mean latency, "
      f"and {best['prefill_saved']:.0%} of prefill eliminated.")

Ablation: contribution of each mechanism

configuration                 steps    tput   mean lat   p99 lat  prefill saved
--------------------------------------------------------------------------------
static batching               4,741    15.3        570      2096          0.0%
+ continuous batching         4,707    15.4        540      2063          0.0%
+ prefix caching              4,707    15.4        540      2063         77.6%

End to end: 1.01x fewer steps, 1.01x throughput,
1.06x lower mean latency, and 78% of prefill eliminated.


In [18]:
# Memory pressure: what happens when blocks run short
print("\nEffect of KV memory on achievable throughput:\n")
print(f"{'KV blocks':>11} {'max concurrent':>16} {'throughput':>12} {'p99 latency':>13}")
print("-" * 56)
for blocks in (256, 512, 1024, 2048, 4096):
    server = ToyServer(ServerConfig(num_kv_blocks=blocks))
    for request in build_workload():
        server.submit(request)
    m = server.run()
    peak = m['throughput']
    print(f"{blocks:>11} {blocks*16:>16,} {peak:>12.1f} {m['p99_latency']:>13.0f}")

print("\nMore KV memory means more concurrent sequences, which means a larger")
print("batch, which -- per notebook 21 -- means more throughput. This is the")
print("chain that makes PagedAttention a THROUGHPUT optimization and not just")
print("a memory one.")


Effect of KV memory on achievable throughput:

  KV blocks   max concurrent   throughput   p99 latency
--------------------------------------------------------
        256            4,096          7.5          9645
        512            8,192         14.3          4758
       1024           16,384         15.4          2556
       2048           32,768         15.4          2063
       4096           65,536         15.4          2063

More KV memory means more concurrent sequences, which means a larger
batch, which -- per notebook 21 -- means more throughput. This is the
chain that makes PagedAttention a THROUGHPUT optimization and not just
a memory one.


## 9. Constrained decoding

A different requirement that lands in the serving layer. Applications often need output matching a
schema — valid JSON, a specific enum, a function-call signature.

Prompting for it is unreliable. **Constrained decoding** makes invalid output *impossible*: compile
the grammar into a state machine, and at each step mask the logits of every token that cannot
appear next.

The implementation cost is that the mask must be computed per step per sequence. Fast
implementations (Outlines, XGrammar) precompute token masks per automaton state so the runtime work
is a table lookup.

In [19]:
class JSONStateMachine:
    """
    A miniature automaton for `{"key": <number>}`.

    Real grammar engines compile arbitrary CFGs; the principle is identical --
    the state determines which tokens are legal, and the rest are masked to
    -inf before sampling.
    """

    STATES = {
        'start': ['{'],
        'after_open': ['"'],
        'in_key': ['k', 'e', 'y', '"'],
        'after_key': [':'],
        'value': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
        'after_value': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '}'],
        'done': [],
    }

    def __init__(self):
        self.state = 'start'

    def allowed(self):
        return self.STATES[self.state]

    def advance(self, char):
        transitions = {
            ('start', '{'): 'after_open',
            ('after_open', '"'): 'in_key',
            ('after_key', ':'): 'value',
        }
        if (self.state, char) in transitions:
            self.state = transitions[(self.state, char)]
        elif self.state == 'in_key':
            self.state = 'after_key' if char == '"' else 'in_key'
        elif self.state in ('value', 'after_value'):
            self.state = 'done' if char == '}' else 'after_value'
        return self.state


ALPHABET = list('{}":,0123456789key ')
CHAR_TO_ID = {c: i for i, c in enumerate(ALPHABET)}


def constrained_generate(logit_fn, max_len=20):
    """Generate under grammar constraints by masking illegal tokens."""
    machine = JSONStateMachine()
    output = []
    for _ in range(max_len):
        allowed = machine.allowed()
        if not allowed:
            break
        logits = logit_fn(output)
        mask = torch.full((len(ALPHABET),), float('-inf'))
        for char in allowed:
            mask[CHAR_TO_ID[char]] = 0.0
        token = int((logits + mask).argmax())
        char = ALPHABET[token]
        output.append(char)
        machine.advance(char)
    return ''.join(output)


# A deliberately uncooperative "model" that outputs random preferences
torch.manual_seed(3)
random_logits = lambda history: torch.randn(len(ALPHABET))

print("Constrained generation from a RANDOM model:\n")
for trial in range(5):
    print(f"  {constrained_generate(random_logits)!r}")

print("\nEvery output is structurally valid JSON despite the model producing")
print("random preferences. Validity is enforced by the sampler, not hoped for")
print("from the prompt.")
print("\nThe caveat worth knowing: constraining changes the distribution. If the")
print("model wanted to refuse, or the schema does not fit the answer, you get a")
print("well-formed WRONG answer. Constraints guarantee syntax, never semantics.")

Constrained generation from a RANDOM model:

  '{"y":333696}'
  '{"e":2925191249426}'
  '{"yykky":22194977179'
  '{"yyy":571841789}'
  '{"":6}'

Every output is structurally valid JSON despite the model producing
random preferences. Validity is enforced by the sampler, not hoped for
from the prompt.

The caveat worth knowing: constraining changes the distribution. If the
model wanted to refuse, or the schema does not fit the answer, you get a
well-formed WRONG answer. Constraints guarantee syntax, never semantics.


## 10. Real systems

| System | Distinguishing design |
|---|---|
| **vLLM** | Originated PagedAttention. Broad model coverage, continuous batching, chunked prefill, prefix caching. The default choice. |
| **SGLang** | RadixAttention (prefix caching as a first-class primitive) plus a frontend language for structured generation. Strongest on multi-turn and agentic workloads with heavy prefix sharing. |
| **TensorRT-LLM** | Ahead-of-time compiled kernels for NVIDIA hardware. Fastest per-GPU when your model and shapes are supported; least flexible. |
| **llama.cpp** | CPU and consumer-GPU focus, GGUF quantization, aggressive memory-mapping. The local-inference standard. |
| **Hugging Face TGI** | Production-oriented serving with strong observability and safetensors integration. |

They differ far less than they used to — continuous batching and paged KV are table stakes now.
The remaining differentiators are prefix-cache sophistication, quantization support, and how
well they handle structured output.

## Summary

Every technique here follows from notebook 21's result that decode is memory-bound and needs
large batches.

**Static batching wastes most of the machine.** With realistic long-tailed output lengths we
measured utilization well under half: the batch cannot retire until its slowest member finishes,
and new requests cannot join.

**Continuous batching** schedules per *iteration* — retire and admit after every forward pass. Pure
scheduling, no kernel changes, and the advantage *grows* with batch size because a bigger batch is
more likely to contain one very long request.

**PagedAttention** replaces contiguous per-sequence reservations with fixed-size blocks and a block
table. Fragmentation drops from `max_len − actual_len` to at most one partial block, which fit many
times more sequences in the same memory in our measurement. Reference counting gives copy-on-write,
so `n=8` sampling of a long prompt costs barely more than `n=1`.

**Prefix caching** shares blocks *across* requests via a radix tree. On a workload with a shared
system prompt we measured a large fraction of prefill eliminated, and in multi-turn chat the hit
rate climbs with every turn. Since prefill is the compute-bound phase, this directly cuts TTFT.

**Chunked prefill** interleaves prompt processing with decode so a long prefill does not stall every
streaming user — and as a bonus mixes a compute-bound phase with a memory-bound one.

**Speculative decoding is exact.** We verified empirically that the accept/reject rule reproduces
the *target* distribution even with a poor draft model driving acceptance to a low rate. A bad
draft costs speed, never quality. There is an optimal draft length, and it grows with acceptance.

**Preemption prefers recomputation** over swapping, because prefill is fast and PCIe is not.
**Disaggregation** separates the two phases onto hardware suited to each.

### Key Takeaways

1. **Serving is a batching problem.** Notebook 21 said throughput needs large batches; everything
   here is about achieving them.
2. **Continuous batching is the highest-leverage change**, and it is pure scheduling.
3. **PagedAttention is a throughput optimization**, not just a memory one: less fragmentation →
   bigger batch → more throughput.
4. **Copy-on-write makes parallel sampling nearly free.**
5. **Prefix caching pays off most where traffic repeats** — system prompts, few-shot blocks,
   multi-turn chat. That is most production traffic.
6. **Speculative decoding preserves the output distribution exactly.** It spends idle FLOPs, which
   only exist because decode is memory-bound.
7. **N-gram drafting needs no model** and works well when output quotes input.
8. **Recompute, don't swap.** Prefill is one fast compute-bound pass.
9. **Constrained decoding guarantees syntax, never semantics.**

### Self-check

- Why does static batching waste so much with long-tailed output lengths?
- What exactly does continuous batching change, and why does its advantage grow with batch size?
- What is the maximum internal fragmentation per sequence under PagedAttention?
- Explain the chain from KV fragmentation to throughput.
- How does copy-on-write make `n=8` sampling cheap?
- Why does prefix caching improve TTFT specifically rather than TPOT?
- Show that speculative decoding samples from the target distribution. What happens with a bad
  draft model?
- Why is there an optimal draft length `γ`?
- Why prefer recomputation over swapping on preemption?
- Your constrained decoder always emits valid JSON but the values are wrong. Explain.

### What's next

Notebook 24 begins Track C: **retrieval-augmented generation** — the first step in building
applications on top of the systems we have now built.

### References

- Kwon et al., 2023 — [Efficient Memory Management for LLM Serving with PagedAttention (vLLM)](https://arxiv.org/abs/2309.06180)
- Yu et al., 2022 — [Orca: A Distributed Serving System for Transformer-Based Generative Models](https://www.usenix.org/conference/osdi22/presentation/yu) (continuous batching)
- Zheng et al., 2023 — [SGLang / RadixAttention](https://arxiv.org/abs/2312.07104)
- Agrawal et al., 2023 — [SARATHI: chunked prefill](https://arxiv.org/abs/2308.16369) · [Sarathi-Serve](https://arxiv.org/abs/2403.02310)
- Leviathan et al., 2022 — [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)
- Chen et al., 2023 — [Accelerating LLM Decoding with Speculative Sampling](https://arxiv.org/abs/2302.01318)
- Cai et al., 2024 — [Medusa](https://arxiv.org/abs/2401.10774) · Li et al., 2024 — [EAGLE](https://arxiv.org/abs/2401.15077)
- Zhong et al., 2024 — [DistServe: disaggregating prefill and decoding](https://arxiv.org/abs/2401.09670)
- Willard & Louf, 2023 — [Efficient Guided Generation (Outlines)](https://arxiv.org/abs/2307.09702)
- Pope et al., 2022 — [Efficiently Scaling Transformer Inference](https://arxiv.org/abs/2211.05102)